# Step 0 — build the canonical dataset (CPU runtime, run ONCE)

Runtime > Change runtime type > **CPU**. This uses no GPU quota.

Output: `busi_yolo.zip` + its md5. Every training lane downloads that exact zip.
Nobody ever runs this notebook again.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/busi-repro/raw
!pip -q install kaggle opencv-python-headless pyyaml

## Get BUSI

**The Kaggle API works without phone verification.** Phone verification gates
notebook internet and accelerators, not dataset downloads via an API token.

Get the token: kaggle.com > your avatar > Settings > API > **Create New Token**.
That downloads `kaggle.json`. Upload it when the next cell prompts.

If the API refuses for any reason, skip to the manual cell below.

In [ ]:
# --- Option A: Kaggle API
from google.colab import files
import os, pathlib
if not pathlib.Path('/root/.kaggle/kaggle.json').exists():
    files.upload()                      # pick kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    !mv kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json

SLUG = 'aryashah2k/breast-ultrasound-images-dataset'   # original BUSI upload
# SLUG = 'sabahesaraki/breast-ultrasound-images-dataset'  # the one the paper cites
!kaggle datasets download -d {SLUG} -p /content/drive/MyDrive/busi-repro/raw --force

In [ ]:
# --- Option B: manual. Download the zip in your browser, drop it in
#     Drive > busi-repro > raw/, then just run the unzip cell below.
!ls -lh /content/drive/MyDrive/busi-repro/raw/

In [ ]:
!mkdir -p /content/busi_raw
!unzip -q -o /content/drive/MyDrive/busi-repro/raw/*.zip -d /content/busi_raw
# Sanity check the layout before building anything.
!find /content/busi_raw -maxdepth 3 -type d
!find /content/busi_raw -name '*.png' | wc -l    # expect ~1578 (780 images + masks)

In [ ]:
%cd /content
![ -d busi-repro ] || git clone -q https://github.com/YOUR_USER/busi-repro.git
%cd /content/busi-repro && git pull -q

# --src can point anywhere above the class folders; the script locates them.
!python -m src.build_dataset --src /content/busi_raw --out /content/busi_yolo

**Read the output of that cell before continuing.** You want:
- ~780 kept, split 56.0 / 26.9 / 17.1 percent
- test totals 66 benign / 31 malignant / 20 normal = **117** (matches the paper's Fig. 6)
- train 306 per class after oversampling = 918
- the near-duplicate count, and whether any pair straddles the test boundary

**Copy the printed md5.** It goes in every lane notebook.

In [ ]:
# Publish the processed zip + the reports.
!cp /content/busi_yolo.zip /content/drive/MyDrive/busi-repro/
!cp /content/busi_yolo/manifest.csv /content/busi_yolo/dedup_report.csv \
    /content/drive/MyDrive/busi-repro/
!md5sum /content/busi_yolo.zip
!du -h /content/busi_yolo.zip

In [ ]:
# MedSAM embedding cache for the 117 test images. CPU, ~10 min, no GPU quota.
!pip -q install 'transformers>=4.44' torch --no-warn-conflicts
!python -m src.medsam cache --images /content/busi_yolo/test/images \
    --out /content/drive/MyDrive/busi-repro/medsam_cache

In [ ]:
# Section 4.3 oracle numbers. Independent of YOLO, so it can run right now.
!python -m src.medsam standalone --data /content/busi_yolo \
    --cache /content/drive/MyDrive/busi-repro/medsam_cache \
    --src /content/busi_raw \
    --out /content/drive/MyDrive/busi-repro/results/medsam_standalone.json